In [1]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
# model_name = "quwsarohi/NanoAgent-135M-8bit"
# model_name = "/Users/ohi/Documents/GitHub/NanoAgent/weights/NanoAgent-135M-8bit"
# model_name = "PleIAs/Monad"
# model_name = "../weights/NanoAgent-135M-grpo-web"
model_name = "../weights/NanoAgent-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token_id = 3

def inference(messages, max_new_tokens=256, temperature=0.0, min_p=0.15, **kwargs):
    if isinstance(messages, list):
        continue_final_message = messages[-1]["role"] == "assistant"
        messages = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=not continue_final_message,
            continue_final_message=continue_final_message,
        )
    inputs = tokenizer.encode(messages, return_tensors="pt")
    outputs = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True if temperature > 0 else False,
        min_p=0.15 if temperature > 0 else None,
        temperature=temperature if temperature > 0 else None,
        **kwargs,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1] :], skip_special_tokens=True)

In [5]:
sys_prompt = """You are a helpful assistant that answers in JSON. Here's the json schema you must adhere to:
<schema>
{'UserQuery': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'The natural language query from the user.'}, 'context': {'type': 'object', 'additionalProperties': {'type': 'string'}, 'description': "The context of the user's query including account type and device used."}, 'expected_response': {'type': 'object', 'properties': {'information': {'type': 'string', 'description': 'The information to be provided to the user as part of the response.'}, 'action': {'type': 'string', 'description': 'The action the user should take following the information provided.'}, 'link': {'type': 'string', 'format': 'uri', 'description': 'A link to a webpage where the user can perform the action described.'}}, 'required': ['information', 'action', 'link'], 'description': 'The structured response expected by the user, including information and action.'}}, 'required': ['query', 'context', 'expected_response']}}
</schema>
"""

user_prompt = """I am working on integrating an AI-driven Simulacra Agent into our customer support system to process user queries more efficiently. The agent should be capable of understanding user queries in natural language, identifying the context of the query, and generating a structured JSON response that includes both the information requested and the action to be taken. For example, if a user asks 'How do I reset my password?', the agent should recognize the intent to reset a password and provide a JSON object with instructions and a link to the password reset page. The context might include the user's account type, which is 'premium', and the device they are using, which is 'mobile'. The expected JSON response should provide the information 'To reset your password, please follow the instructions on the password reset page.', the action 'Click the link below to navigate to the password reset page.', and include the link 'https://www.example.com/password-reset'. The JSON object that we are aiming to generate would look something like this: {"UserQuery": {"query": "How do I reset my password?", "context": {"account_type": "premium", "device": "mobile"}, "expected_response": {"information": "To reset your password, please follow the instructions on the password reset page.", "action": "Click the link below to navigate to the password reset page.", "link": "https://www.example.com/password-reset"}}}. This structured response will enable our Simulacra Agent to interact with users in a more human-like and efficient manner, providing them with the exact information they need, along with the necessary actions they should take, all within the context of their current situation."""

messages = [
    {"role": "system", "content": sys_prompt},
    {"role": "user", "content": user_prompt},
    # {"role": "assistant", "content": "```json\n"}
]

print(inference(messages, max_new_tokens=128, temperature=0.3))

```json
{
  "UserQuery": {
    "query": "How do I reset my password?",
    "context": {
      "account_type": "premium",
      "device": "mobile"
    },
    "expected_response": {
      "information": "To reset your password, please follow the instructions on the password reset page.",
      "action": "Click the link below to navigate to the password reset page.",
      "link": "https://www.example.com/password-reset"
    }
  },
  "required": ["query", "context", "expected_response"],
  "


In [11]:
# messages = [{"role": "user", "content": "In a communication system, data is sent using orthogonal Latin squares. The transmission is organized into 4 time slots. Each slot carries 3 distinct symbols. During the transmission, an additional 4 symbols are sent from a separate source. What is the total number of symbols sent during this transmission?"}]
messages = [{"role": "user", "content": "-2*4=?"}]
print(inference(messages, max_new_tokens=128, temperature=0.3))

To solve this equation, we need to isolate the variable 'x' on one side of the equation.

First, let's move the constant term '2' to one side of the equation:

-2*4 = -2*4 + 4
-2*4 = -4 + 4
-2*4 = 4

Now, we need to move the constant term '4' to the other side of the equation:

-2*4 = -4 + 4
-2*4 = -8

Finally, we need to move the variable 'x


In [10]:
# # messages = [{"role": "user", "content": "Hi!"}]
# input_text = \
# """<|im_start|>user
# Hi!<|im_end|>
# <|im_start|>assistant
# """
# print(inference(input_text, max_new_tokens=1024))

In [4]:
TOOLS = [
    {
        "name": "web_search",
        "description": "Performs a web search for a query and returns a string of the top search results formatted as markdown with titles, links, and descriptions.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to perform.",
                }
            },
            "required": ["query"],
        },
    },
    {
        "name": "visit_webpage",
        "description": "Fetches and displays the textual content of a webpage (converted to Markdown) from a given URL.",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "The full URL of the webpage to retrieve.",
                }
            },
            "required": ["url"],
        },
    },
    {
        "name": "final_answer",
        "description": "Provides a final answer to the given problem.",
        "parameters": {
            "type": "object",
            "properties": {
                "answer": {
                    "type": "string",
                    "description": "The final answer to the problem",
                }
            },
            "required": ["answer"],
        },
    },
]

TOOL_TEMPLATE = """You are a helpful AI assistant. You have a set of possible functions/tools inside <tools></tools> tags. 
Based on question, you may need to make one or more function/tool calls to answer user.

You have access to the following tools/functions:
<tools>{tools}</tools>

For each function call, return a JSON list object with function name and arguments within <tool_call></tool_call> tags."""


messages = [
    {"role": "system", "content": TOOL_TEMPLATE.format(tools=json.dumps(TOOLS))},
    {"role": "user", "content": "What capabilities do you have?"},
    # {"role": "assistant", "content": "<think>"}
]

print(inference(messages))

I have the following capabilities:
1. **web_search** function to perform a web search for a query and return a string of the top search results formatted as markdown with titles, links, and descriptions.
2. **visit_webpage** function to fetch and display the textual content of a webpage from a given URL.



In [5]:
from ddgs import DDGS

def web_search(query: str, max_results: int = 2) -> str:
    """
    Performs a DuckDuckGo web search and returns formatted markdown output.
    
    Args:
        query (str): Search query string.
        max_results (int): Number of results to return.
    
    Returns:
        str: Markdown formatted search results.
    """
    results = DDGS().text(query, max_results=max_results)
    
    output_lines = ["## Search Results"]
    
    for i, item in enumerate(results):
        title = item.get("title", "No Title")
        href = item.get("href", "")
        snippet = item.get("body", "")
        date = item.get("date", "Unknown date")
        
        output_lines.append(
            f"{i}. [{title}]({href})\n"
            f"Date published: {date}\n\n"
            f"{snippet}\n"
        )
    
    return "\n".join(output_lines)

In [6]:
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md

def visit_webpage(url: str, main_selector: str = None) -> str:
    """
    Fetch a URL, extract HTML content, convert to Markdown, and return it.
    
    Args:
        url (str): The webpage URL.
        main_selector (str): Optional CSS selector to extract specific content
                             (e.g. "#content", ".article", "main", etc.)
    
    Returns:
        str: Markdown text extracted from the page.
    """
    
    # Fetch page
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    
    # Parse HTML
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Extract main content if selector provided
    if main_selector:
        element = soup.select_one(main_selector)
        if not element:
            raise ValueError(f"No element found for selector: {main_selector}")
        html_content = str(element)
    else:
        # Fallback: use <body>
        html_content = str(soup.body or soup)
    
    # Convert HTML → Markdown
    markdown_text = md(html_content, heading_style="ATX")

    return markdown_text

In [7]:
def wiki_search(inp: str):
    import wikipedia

    result = None
    try:
        result = wikipedia.summary(inp)
    except wikipedia.exceptions.DisambiguationError as e:
        options = e.options
        results = []
        for option in options:
            results.append(f"# Topic: {option}\n{wikipedia.summary(option)}")
        result = "\n".join(results)
    return json.dumps({"search_result": result.strip()})


def extract_toolcall(input_str):
    import re

    calls = re.findall(r"<tool_call>(.*?)</tool_call>", input_str, re.DOTALL)
    if calls:
        calls = calls[-1].strip()
        try:
            return json.loads(calls)
        except:
            return []
    return []


# messages = [
#     {
#         "role": "system",
#         "content": TOOL_TEMPLATE.format(tools=json.dumps(TOOLS))
#         #+ "Think step by step before answering.",
#     },
#     {
#         "role": "user",
#         # "content": "Hey, what is the scientific name of yellow toadflax?",
#         "content": "When Dr Yunus he became elected as president??"
#     },
#     # {"role": "assistant", "content": "<think>"}
# ]

# print("User:", messages[-1]["content"])
# llm_tool_call = inference(messages, max_new_tokens=512)
# print("LLM:", llm_tool_call)

# search_query = extract_toolcall(llm_tool_call)[0]["arguments"]["query"]
# # search_result = wiki_search(search_query)
# search_result = web_search(search_query, max_results=4)
# tool_reply = f"<tool_response>{search_result}</tool_response>"

# print("--- Web Result ---")
# print(search_result)
# print("-------------------\n")

# messages += [
#     {"role": "assistant", "content": llm_tool_call},
#     {"role": "user", "content": tool_reply},
# ]

# ret = inference(messages, max_new_tokens=512)
# print("LLM:", ret)

In [15]:
def iterate_tools(user_question):
    messages = [
        {
            "role": "system",
            "content": TOOL_TEMPLATE.format(tools=json.dumps(TOOLS))
        },
        {
            "role": "user",
            "content": user_question
        },
    ]
    tool_call_lead = [{"role": "assistant", "content": "<tool_call>["}]
    print(tokenizer.apply_chat_template(messages, tokenize=False))

    for itr in range(4):
        # llm_gen = "<tool_call>[" + inference(messages + tool_call_lead, max_new_tokens=512)
        llm_gen = inference(messages, max_new_tokens=512, temperature=0.3, min_p=0.1)
        messages.append({"role": "assistant", "content": llm_gen})
        print("LLM:", llm_gen)

        try:
        # if True:
            gen_tool = extract_toolcall(llm_gen)[0]
            if 'web_search' == gen_tool['name']:
                result = web_search(gen_tool['arguments']['query'], max_results=5)
            elif 'final_answer' in gen_tool['name']:
                result = gen_tool['arguments']['answer']
                return result
            elif 'visit_webpage' in gen_tool['name']:
                result = visit_webpage(gen_tool['arguments']['url'])
            
            tool_reply = f"<tool_response>{result}</tool_response>"
            messages.append({"role": "user", "content": tool_reply})
            print("## Tool reply:\n", tool_reply)
        except:
            pass
    
    return llm_gen

iterate_tools("Hey, what is the scientific name of yellow toadflax?")
# iterate_tools("Hey, what is the weather in ottawa?")
# iterate_tools("What ingerients are needed to make a cake?")

<|im_start|>system
You are a helpful AI assistant. You have a set of possible functions/tools inside <tools></tools> tags. 
Based on question, you may need to make one or more function/tool calls to answer user.

You have access to the following tools/functions:
<tools>[{"name": "web_search", "description": "Performs a web search for a query and returns a string of the top search results formatted as markdown with titles, links, and descriptions.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "The search query to perform."}}, "required": ["query"]}}, {"name": "visit_webpage", "description": "Fetches and displays the textual content of a webpage (converted to Markdown) from a given URL.", "parameters": {"type": "object", "properties": {"url": {"type": "string", "description": "The full URL of the webpage to retrieve."}}, "required": ["url"]}}, {"name": "final_answer", "description": "Provides a final answer to the given problem.", "parameter

'1. [yellow-toadflax-ID](http://mtwow.org/yellow-toadflax-ID.html)\nDate published: Unknown date\n\nPurchasing Insects: You can purchase the yellow toadflax feeding insect ( Brachypterolus pulicarius) at http://www.bio-control.com/ at about $75.00 ...\n\n2. [yellow toadflax | Hiking the GTA](https://hikingthegta.com/tag/yellow-toadflax/)\nDate published: Unknown date\n\nYellow Toadflax ( Linaria vulgaris ), Common Toadflax or Butter-and-eggs is a species of toadflax ( Linaria ), native to most of Europe and northern, central and eastern Asia. It has also ...\n</tool_call>\n'

In [9]:



# # Example usage:
# if __name__ == "__main__":
#     url = "https://www.bbc.com/news/articles/clyg7we8xvno"
#     markdown = visit_webpage(url, main_selector="body")
#     print(markdown)
